# Word Alignment — Pairwise Levenshtein Normalisation

Determines the **best word alignment** for each source word against the DAT and DIT transcripts & uses pairwise Levenshtein normalization (cf. cross-comparison.ipynb).

Scoring methods:
- **Combined Weighted Score**: $\alpha \cdot \text{lev\_sim} + (1-\alpha) \cdot \text{pos\_sim}$, $\alpha = 0.7$
- **Combined Harmonic Score**: $\frac{2 \cdot \text{lev\_sim} \cdot \text{pos\_sim}}{\text{lev\_sim} + \text{pos\_sim}}$

## Setup

In [73]:
import sys
import importlib
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path("__file__").resolve().parent))

import util.utils as utils
importlib.reload(utils)

from util.utils import build_word_comparison_df, clean

In [74]:
PROJECT_ROOT = Path("__file__").resolve().parent.parent
ANALYSIS_DIR = PROJECT_ROOT / "samples"
DIT_TSV = ANALYSIS_DIR / "dialect-ignorant-transcript.tsv"
DAT_TSV = ANALYSIS_DIR / "dialect-aware-transcript.tsv"

## Load Data

In [75]:
df_dit = pd.read_csv(DIT_TSV, sep='\t', encoding='utf-8-sig')
df_dat = pd.read_csv(DAT_TSV, sep='\t', encoding='utf-8-sig')
df = pd.merge(df_dit, df_dat[['path', 'DAT']], on='path', how='inner')

print(f"Clips: {len(df)}")
df.head()

Clips: 100


,path,duration,sentence,sentence_source,client_id,dialect_region,canton,zipcode,age,gender,DIT,DAT
0,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,4.778662,Insgesamt habe ich einige Hunderttausend Frank...,news,300bb931-79ae-40ec-b989-3efd5e83f4c2,Zürich,ZH,8408.0,fourties,male,Insgesamt habe ich einige hundert Dusik vor un...,Insgesamt habe ich einige 100.000 Franken verl...
1,clips\6e084270-8d26-43d9-ba69-5e8ee793ab8c\dca...,5.294150,Welche Rolle hatte Rosenberg während des Holoc...,news,6e084270-8d26-43d9-ba69-5e8ee793ab8c,Zürich,ZG,6340.0,twenties,female,"Weli Rolle, höchter Uz stoodhergewerden von Ho...",Wer ironne hättet viele bands auf den einzugem...
2,clips\6858a37b-edd0-4fdf-871c-96d3b1bd3e21\82f...,5.802676,Das ist angesichts aller schlechten Optionen d...,news,6858a37b-edd0-4fdf-871c-96d3b1bd3e21,Zürich,ZH,8704.0,fourties,male,Das ist angesichts vor allen Schlachten Option...,Das ist angesichts von allen schlechten Option...
3,clips\c4c03e6f-50a2-4d24-ae88-caf3032798fa\a21...,5.973333,Ebenfalls keinen Sieg durfte die AC Milan feiern.,news,c4c03e6f-50a2-4d24-ae88-caf3032798fa,Zürich,AG,4663.0,fourties,male,Aber voll ScreenZig hat 100 000 derumphierig,Aber falls kein Sieg hat AC Milan Dörfe 4.
4,clips\f877ee50-af2b-423b-bdda-0cc406032c45\4e5...,4.778662,Viele Fussballfans haben die Alte Reithalle un...,parliament,f877ee50-af2b-423b-bdda-0cc406032c45,Zürich,ZH,8405.0,fourties,female,Viele Fußballfans hängt die Alldri-Taller unte...,Viele Fußballfans handt die All 3 Taller unter...


## Cross-Comparison

In [76]:
all_words = (
    df['sentence'].dropna().str.split().explode().tolist()
    + df['DIT'].dropna().str.split().explode().tolist()
    + df['DAT'].dropna().str.split().explode().tolist()
)

all_sentences = pd.concat([
    df['sentence'].dropna(),
    df['DIT'].dropna(),
    df['DAT'].dropna()
])

global_max_word_length     = max(len(clean(w)) for w in all_words)
global_max_sentence_length = max(len(str(s).split()) for s in all_sentences)

print(f"Global max word length     (Levenshtein normaliser): {global_max_word_length}")
print(f"Global max sentence length (position normaliser):    {global_max_sentence_length}")

Global max word length     (Levenshtein normaliser): 27
Global max sentence length (position normaliser):    65
